# Basic Usage

For plotting images with widgets and callbacks for interactivity.

In [ ]:
import numpy as np
import panel as pn

from libertem_ui.figure import ApertureFigure
from libertem_ui.display.cursor import Cursor

## Overview

`LiberTEM-panel-ui` is based on two libraries:

- `bokeh`, which provides a fairly low-level plotting interface (figure, axes, glyphs etc) and infrastructure for interactivity in a browser
- `panel`, which itself is based on `bokeh`, which provides a higher-level interface for widgets, callbacks and easy display of `bokeh` components in Jupyter Notebooks as well as in a browser. It also provides a good set of template, themeing and layout infrastrucure, as well as advanced widgets like code editors, 3D display or interactive tools for tabular data.

`LiberTEM-panel-ui` provides two main features:

- `libertem_ui.figure.ApertureFigure`, which displays various sorts of *image-like* data on a `bokeh.plotting.Figure`, and provides some useful convenience features on this figure, such as adjustable colormaps, automatic downsampling, and mask / ROI tools
- Under `libertem_ui.display`, various wrappers for `bokeh.models.Glyph`, i.e. things which can be plotted on a figure axis like a set of points, polygons, lines etc. The added value is:
  - Simplified constructors
  - Unified creation of the data source, the glyph(s), and adding them to one-or-more figures, to reduce boilerplate
  - Common interface for updating data
  - Easy creation of tools (such as click to create a point)
  - Structure to handle groups of `Glyph` as one object which share the same data source and methods

Otherwise, the features are quite "thin", and there is no attempt to re-define the styling or callback infrastructre of `bokeh` or `panel`. If you want to modify a property of the axes, for example, you do it via the `bokeh` API on the `ap_fig.fig` property, for example:

```python
ap_fig.fig.xaxis.axis_label = "Width"
```

If you want to trigger a callback when the data of a figure changes, you do it directly through `bokeh` with `data_source.on_change(...)`, for example.

## Simple display and callbacks

To display an image on a new figure (or stack, complex image etc) create a new `ApertureFigure`. To cause it to display after a notebook cell the `fig.layout` property must be returned from that cell. `.layout` is a `panel.layouts.Column` layout which contains the figure and its support widgets. The cell that defines the figure and the cell which displays it need not be the same.

In [ ]:
fig1 = ApertureFigure.new(np.random.uniform(size=(32, 64)), title=f"A random image")
fig1.layout

We can update the image data by defining a widget to trigger a callback, then calling the figure's `update` method:

In [ ]:
im_shape = (32, 64)
fig2 = ApertureFigure.new(np.random.uniform(size=im_shape), title=f"An updating image")

def update_image(event):
    fig2.update(np.random.uniform(size=im_shape))

btn = pn.widgets.Button(name="Update figure")
btn.on_click(update_image)
# Since we already have a Column layout for the figure, use it rather than creating our own
fig2.layout.insert(0, btn)
fig2.layout

A `Button` has a simplified `on_click` callback endpoint, more general widgets use `widget.param.watch(cb, "property")`, here we use a slider to update the figure title:

In [ ]:
slider = pn.widgets.FloatSlider(name="Figure title", start=0., end=1., step=0.1, value=0.)
fig3 = ApertureFigure.new(np.random.uniform(size=(32, 64)), title=f"Slider value: {slider.value}")

def update_title(event):
    fig3.fig.title.text = f"Slider value: {event.new}"

slider.param.watch(update_title, "value_throttled")
fig3.layout.insert(0, slider)
fig3.layout

## Callback from changing `Glyph` data

Here we add a `libertem_ui.display.cursor.Cursor` to the figure, and update some text in response to the `Cursor` being dragged on screen.

Behind every `Glyph` is a `bokeh` `ColumnDataSource`, which is a wrapper of a dict-like structure with equal-length columns. The `Glyph` combines with the `ColumnDataSource` as a `Renderer` on a figure. If the rendered glyph is updated on screen, the change is synchronised back to Python, and if the `ColumnDataSource` is updated on the Python side, the change is synchronised to the displayed renderer. This update can trigger callbacks both in the browser as Javascript, or in Python.

A `Cursor` (and other components in `libertem_ui.display`) is a wrapper around `Glyph(s)`, a `ColumDataSource` and one-or-more `Renderers` for one-or-more figures. The data source is always available under `cursor.cds` so that callbacks can be registered.

The `Glyph` itself sets visual properties of something to be rendered, e.g. colour. It is usually accessible under `cursor.glyph`, though this interface needs to be unified across components.

In [ ]:
fig4 = ApertureFigure.new(np.arange(64 * 64).reshape(64, 64), title=f"Has a cursor")
# Add a "Cursor" to the figure on top of the image
# which is a single point with an X/Y position and convenience methods
cursor = (
    Cursor
    .new()
    .from_pos(x=32, y=32)
    .on(fig4.fig)
    .editable(selected=True)  # this adds the necessary components to make the cursor draggable
)
# The cursor Glyph is accessed via the .cursor property, until the API is unified
cursor.glyph.line_color = "red"
title_str = lambda cur: f"Cursor position: x={cur.current_pos().x:.1f}, y={cur.current_pos().y:.1f}"
label = pn.widgets.StaticText(value=title_str(cursor))

def update_label(attr, old, new):
    # This is a "bokeh-style" callback because it will trigger directly from a ColumnDataSource
    # The callback must have three arguments [attr, old, new]
    # attr will be "data"
    # old will be the CDS dict before the trigger
    # new will be the CDS dict after the trigger
    label.value = title_str(cursor)

# trigger whenever the "data" attribute of the cursor ColumnDataSource changes
cursor.cds.on_change("data", update_label)

fig4.layout.insert(0, label)
fig4.layout

## List of features



## Use as a UDF Live Plot

We can also use the `ApertureFigure` as a UDF Live Plot, through the specialised subclass `libertem_ui.live_plot.AperturePlot`.

In [ ]:
import libertem.api as lt
from libertem.udf.sumsigudf import SumSigUDF
from libertem.udf.sum import SumUDF
from libertem_ui.live_plot import AperturePlot

In [ ]:
ctx = lt.Context.make_with("inline")
ds_shape = (5, 7, 8, 8)
ds = ctx.load("memory", data=np.arange(np.prod(ds_shape)).reshape(ds_shape), num_partitions=4)
ds.shape

Define a convenience UDF to show sequential updating partition-by-partition:

In [ ]:
import time

class GoSlowUDF(SumUDF):
    def process_tile(self, tile):
        time.sleep(0.25)
        return super().process_tile(tile)

As is common in notebooks, we need to declare and display the UI before actually running the computation. This also demonstrates how to layout multiple figures in the same cell output:

In [ ]:
sumsig_udf = SumSigUDF()
sumsig_plot = AperturePlot.new(ds, sumsig_udf, title="Sum-over-sig")
sum_udf = SumUDF()
sum_plot = AperturePlot.new(ds, sum_udf, title="Sum-over-nav")
pn.layout.Row(
    sumsig_plot.layout,
    sum_plot.layout,
)


Once displayed, we can run the computation itself

In [ ]:
_ = ctx.run_udf(ds, [GoSlowUDF(), sumsig_udf, sum_udf], plots=[sumsig_plot, sum_plot])

We can use plot tools to define an interactive ROI, by calling `ApertureFigure.add_mask_tools()`:

In [ ]:
sumsig_plot = AperturePlot.new(ds, sumsig_udf, title="Sum-over-sig")
sumsig_plot.add_mask_tools(activate=True)
sum_plot = AperturePlot.new(ds, sum_udf, title="Sum-over-nav")
pn.layout.Row(
    sumsig_plot.layout,
    sum_plot.layout,
)

We can then call `figure.get_mask(ds.shape.nav)` to get a boolean mask which we can supply to `ctx.run_udf`:

In [ ]:
_ = ctx.run_udf(
    ds,
    [GoSlowUDF(), sumsig_udf, sum_udf],
    plots=[sumsig_plot, sum_plot],
    roi=sumsig_plot.get_mask(ds.shape.nav)
)

The call to `run_udf` could be made from a callback, which lets us trigger the run with the ROI from the interface itself. Care needs to be taken, however, with long-running UDFs which block computation - blocking execution within an interactive cell is a bit of a gray area, ideally solved with `async` but this requires more reseearch to implement robustly with Panel.

## Beyond image display

We can also use Bokeh to display non-image information, and via the `libertem_ui` interface update the display easily.

In [ ]:
from bokeh.plotting import figure
from libertem_ui.display.points import PointSet
from libertem_ui.display.lines import Curve

We will create a scatter plot (`PointSet`) and line plot (`Curve`) which share the same data source. Updating one component will also update the other.

This example also uses a plain Bokeh `Figure`, rather than a wrapped `ApertureFigure`.

In [ ]:
xvals = np.arange(10)
yvals = np.random.uniform(-0.1, 0.1) + 0.5 * xvals

fig5 = figure(title="A line")
fig5.frame_height = 400
fig5.frame_width = 600
# Creating the PointSet internally creates the ColumnDataSource with default column names
points = PointSet.new().from_vectors(xvals, yvals).on(fig5)
# create the `Curve` object manually using the existing data source,
# because `PointSet` uses different column names we need to override `xkey` and `ykey`
line = Curve(points.cds, xkey=points.glyph.x, ykey=points.glyph.y).on(fig5)

def update_data(event):
    # For the moment we must re-supply both columns
    # In the future should be able to update x or y only
    points.update(x=xvals, y=np.random.uniform(-0.1, 0.1) + 0.5 * xvals)

update_btn = pn.widgets.Button(name="New data")
update_btn.on_click(update_data)

pn.layout.Column(
    update_btn,
    # to get the Bokeh Figure to display in a notebook
    # we need to wrap it in a Panel `Bokeh` pane
    pn.pane.Bokeh(fig5),
)

## Linked concentric `RingSet` with scale slider

This is an advanced example that creates concentric rings whose centre point can be interactively dragged, and whose initial radii can be scaled using a slider.

To ensure concentricity each ring is a separate `Glyph`, all rings share the `cx`, `cy` columns of the shared data source, while each ring has its own radii columns.

A `ColumnDataSource` normally requires all columns to have the same length, so to define 3 rings with 3 inner and outer radii each, we also must define three centre coordinates:

```
cx, cx, ri, ro
--------------
32, 32, 10, 15
32, 32, 20, 25
32, 32, 30, 35
```

the implication here is that we then need to manually synchronise the centre coordinate on every change if we want the rings to stay concentric.

Instead we can use the following data structure:

```
cx, cx, ri0, ro0, ri1, ro1, ri2, ro2
------------------------------------
32, 32,  10,  15,  20,  25,  30,  35
```

where each ring has its own independent radii columns, and all share `cx, cy`. Updating `cx` or `cy` will change the centre of all rings, while the radii can change independently.

In [ ]:
from libertem_ui.display.points import RingSet

im_shape = (64, 64)
fig2 = ApertureFigure.new(np.zeros(im_shape), title=f"A multi-ringset image")

num_rings = 3
inner_radius = np.arange(1, num_rings + 1) * 4
outer_radius = 1 + (np.arange(1, num_rings + 1) * 4)

def create_rings(fig, cx: float, cy: float, ri: np.ndarray, ro: np.ndarray):
    assert len(ri) == len(ro)
    # The Cursor object acts as a draggable single-point centre marker for the rings
    centre = (
        Cursor
        .new()
        .from_pos(
            x=cx,
            y=cy,
        )
        .on(fig)
        .editable(selected=True)
    )
    centre.glyph.line_color = "red"
    ringsets = []
    for i, (rii, roo) in enumerate(zip(ri, ro)):
        # dynamically add new columns to the data source for each ring and its radii
        rikey = f"ri_{i}"
        rokey = f"ro_{i}"
        centre.cds.data.update({rikey: [rii], rokey:[roo]})
        # create new ringset using the new columns for the radii, and leave `cx/cy` as default keys for the centre
        ringsets.append(
            RingSet(centre.cds, inner_radius=rikey, outer_radius=rokey).on(fig)
        )
    return centre, ringsets

# The initial radii for the rings
ri = (5, 10, 15)
ro = (7, 12, 17)
cursor, ringsets = create_rings(
    fig2.fig, 32, 32, ri, ro,
)

scale_slider = pn.widgets.FloatSlider(
    name="Scale",
    start=0.1,
    end=2.,
    step=0.01,
    value=1.,
)

def update_rings(event):
    # When the slider triggers, scale the initial radii by its value
    new_radii = {}
    for i, (rii, roo) in enumerate(zip(ri, ro)):
        # Dynamically re-generating the key names is brittle,
        # should instead return some kind of setter from create_rings
        new_radii[f"ri_{i}"] = [rii * event.new]
        new_radii[f"ro_{i}"] = [roo * event.new]
    # because we are using Cursor in a non-standard way we
    # need to use `raw_update` to update the extra columns
    cursor.raw_update(**new_radii)

scale_slider.param.watch(update_rings, "value_throttled")
fig2.layout.insert(0, scale_slider)
fig2.layout